#Ingestão e Persistência dos Dados Brutos

In [0]:
# Caminho dos arquivos originais
caminho_origem = "/Volumes/workspace/bronze/olist_raw/"

# Catálogo e schema de destino
catalogo = "workspace"
schema = "bronze"

# Mapeamento dos arquivos CSV para as tabelas Bronze
arquivos = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "category_translation"
}

In [0]:
from pyspark.sql.functions import current_timestamp, lit

for arquivo, tabela in arquivos.items():

    print(f"Iniciando ingestão: {arquivo}")

    # Lê o CSV preservando campos textuais com quebras de linha
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", "UTF-8")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .csv(caminho_origem + arquivo)
    )

    # Adiciona metadados de ingestão
    df = (
        df
        .withColumn("_data_ingestao", current_timestamp())
        .withColumn("_arquivo_origem", lit(arquivo))
    )

    # Nome completo da tabela
    tabela_destino = f"{catalogo}.{schema}.{tabela}"

    # Persistência em Delta Lake
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_destino)
    )

    print(f"Tabela criada: {tabela_destino}")
    print(f"Quantidade de registros: {df.count()}")
    print("----------------------------------")

Iniciando ingestão: olist_customers_dataset.csv
Tabela criada: workspace.bronze.customers
Quantidade de registros: 99441
----------------------------------
Iniciando ingestão: olist_geolocation_dataset.csv
Tabela criada: workspace.bronze.geolocation
Quantidade de registros: 1000163
----------------------------------
Iniciando ingestão: olist_order_items_dataset.csv
Tabela criada: workspace.bronze.order_items
Quantidade de registros: 112650
----------------------------------
Iniciando ingestão: olist_order_payments_dataset.csv
Tabela criada: workspace.bronze.order_payments
Quantidade de registros: 103886
----------------------------------
Iniciando ingestão: olist_order_reviews_dataset.csv
Tabela criada: workspace.bronze.order_reviews
Quantidade de registros: 104162
----------------------------------
Iniciando ingestão: olist_orders_dataset.csv
Tabela criada: workspace.bronze.orders
Quantidade de registros: 99441
----------------------------------
Iniciando ingestão: olist_products_data

In [0]:
display(spark.sql("SHOW TABLES IN workspace.bronze"))

database,tableName,isTemporary
bronze,category_translation,false
bronze,customers,false
bronze,geolocation,false
bronze,order_items,false
bronze,order_payments,false
bronze,order_reviews,false
bronze,orders,false
bronze,products,false
bronze,sellers,false


In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Define o arquivo original de avaliações
arquivo = "olist_order_reviews_dataset.csv"

# Lê o CSV considerando comentários com múltiplas linhas
df_reviews = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("encoding", "UTF-8")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(caminho_origem + arquivo)
)

# Adiciona os metadados de rastreabilidade
df_reviews = (
    df_reviews
    .withColumn("_data_ingestao", current_timestamp())
    .withColumn("_arquivo_origem", lit(arquivo))
)

# Reconstrói a tabela Bronze com a leitura corrigida
(
    df_reviews.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.order_reviews")
)

# Confirma a quantidade de registros carregados
print(f"Registros após correção: {df_reviews.count()}")

Registros após correção: 99224
